# GRPO Training — Qwen2.5-3B-Instruct

**Data Pipeline Incident Response · Meta PyTorch OpenEnv Hackathon**

Two-stage training: SFT warm-start → GRPO reinforcement learning.

- Model: `Qwen/Qwen2.5-3B-Instruct` (text-only, 4-bit quantized)
- Hardware: Kaggle T4 (16GB VRAM)
- Expected runtime: ~15 min SFT + ~45 min GRPO


## 1. Setup

In [1]:
!pip install -q unsloth[colab-new] trl>=0.8.6 peft accelerate bitsandbytes \
    transformers datasets pandas numpy openai python-dotenv
print('Installation complete.')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.
Installation complete.


In [2]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = _s.get_secret('GITHUB_TOKEN')
    HF_TOKEN = _s.get_secret('HF_TOKEN')
except Exception:
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
    HF_TOKEN = os.getenv('HF_TOKEN', '')

REPO_DIR = '/kaggle/working/Meta_hackathon'
if not os.path.exists(REPO_DIR):
    !git clone https://{GITHUB_TOKEN}@github.com/standing-on-giants/Meta_hackathon.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')


Cloning into '/kaggle/working/Meta_hackathon'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 114 (delta 57), reused 86 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 178.91 KiB | 13.76 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/kaggle/working/Meta_hackathon
Working directory: /kaggle/working/Meta_hackathon


In [3]:
import sys, json, textwrap, random, re, torch, numpy as np
from pathlib import Path
sys.path.insert(0, REPO_DIR)
from src.environment import DataPipelineEnv
from src.models import PipelineAction, PipelineObservation
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')


PyTorch 2.10.0+cu128, CUDA: True
GPU: Tesla T4, VRAM: 15.6GB


## 2. Load Qwen2.5-3B (4-bit quantization, LoRA r=32)

In [4]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)
print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters()/1e9:.2f}B')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: Qwen/Qwen2.5-3B-Instruct
Parameters: 3.09B


In [5]:
# LoRA rank=32 (smaller model can afford higher rank)
model = FastLanguageModel.get_peft_model(
    model, r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)')


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable: 59.9M / 1.86B (3.22%)


## 3. Stage 1 — SFT on Gold Trajectories

We train on hard-coded correct action sequences for easy/medium tasks.
This teaches the model output format and basic diagnostic strategy.


In [6]:
SYSTEM_PROMPT = textwrap.dedent('''
You are an expert data engineer diagnosing and fixing broken data pipelines.
You receive pipeline state and must choose ONE action per turn.
WORKFLOW: 1. read_data_sample 2. check_schema/compare_schema 3. Apply fix 4. run_pipeline
RULES: Respond with ONLY a JSON object. Never repeat failing actions. dedup for uniqueness failures.
''').strip()

GOLD_ACTIONS = {
    'easy': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_orders', 'n_rows': 20}},
        {'action_type': 'add_data_filter', 'params': {'step_id': 'transform_orders', 'filter_condition': 'user_id IS NOT NULL'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'medium': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_order_items', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_items', 'patch_type': 'dedup', 'column': 'order_item_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
}

def format_obs(obs, step):
    failed = '\n'.join(f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}' for r in obs.failed_assertions) or '  (none)'
    passed = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag = '\n'.join(f'  {n.step_id}: {n.input_table} -> {n.output_table}' for n in obs.dag_structure)
    hist = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs)
    schema = ''
    if obs.current_schema: schema += '\nSCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff: schema += '\nDIFF: ' + json.dumps(obs.schema_diff)
    sample = ''
    if obs.data_sample: sample = '\nDATA: ' + json.dumps(obs.data_sample[:3], default=str)
    actions = '\n'.join(f'  {a}' for a in obs.actions_taken[-4:]) or '  (none)'
    return f'STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})\nDESCRIPTION: {obs.description}\nPIPELINE PASSED: {obs.pipeline_passed}\nLAST RESULT: {obs.last_action_result}\nDAG:\n{dag}\nFAILING:\n{failed}\nPASSING: {passed}\nHISTORY:\n{hist}\nACTIONS:\n{actions}{sample}{schema}\nRespond with ONE action JSON.'

def collect_gold(task_ids=['easy','medium'], n_ep=10):
    pairs = []
    for tid in task_ids:
        gold = GOLD_ACTIONS.get(tid, [])
        if not gold: continue
        for _ in range(n_ep):
            env = DataPipelineEnv(task_id=tid)
            obs = env.reset()
            for si, ad in enumerate(gold, 1):
                pairs.append((format_obs(obs, si), json.dumps(ad)))
                result = env.step(PipelineAction(**ad))
                obs = result.observation
                if obs.pipeline_passed: break
    return pairs

gold_pairs = collect_gold(n_ep=10)
print(f'Collected {len(gold_pairs)} SFT pairs')
print(f'Sample action: {gold_pairs[0][1]}')


Collected 60 SFT pairs
Sample action: {"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 20}}


In [8]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

sft_texts = [
    tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM_PROMPT},
         {'role':'user','content':obs},
         {'role':'assistant','content':act}],
        tokenize=False, add_generation_prompt=False)
    for obs, act in gold_pairs
]
sft_ds = Dataset.from_dict({'text': sft_texts})

SFT_DIR = '/kaggle/working/sft_qwen'
sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=sft_ds, dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
            average_tokens_across_devices=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=5,  # more epochs for smaller model
        warmup_ratio=0.1, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5, optim='adamw_8bit',
        weight_decay=0.01, lr_scheduler_type='cosine',
        output_dir=SFT_DIR, save_steps=50, seed=42,
    ),
)
print('Starting SFT...')
sft_stats = sft_trainer.train()
print(f'SFT done. Loss: {sft_stats.training_loss:.4f}')
model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/60 [00:00<?, ? examples/s]

Starting SFT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 5 | Total steps = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss
5,2.593291
10,1.507048
15,0.747589
20,0.433179


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sft_qwen/checkpoint-20/tokenizer_config.json.


SFT done. Loss: 1.3203


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/sft_qwen/tokenizer_config.json.


('/kaggle/working/sft_qwen/tokenizer_config.json',
 '/kaggle/working/sft_qwen/chat_template.jinja',
 '/kaggle/working/sft_qwen/tokenizer.json')

## 4. Stage 2 — GRPO Training

Uses environment rewards directly. G=8 completions per prompt.
KL penalty (0.1) anchors to SFT checkpoint to prevent degenerate collapse.


In [9]:
def parse_action(text):
    # Strip <think> tags (Qwen sometimes produces them)
    text = re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()
    if '```' in text:
        text = '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```')).strip()
    start = text.find('{')
    if start == -1: return None
    end = text.rfind('}') + 1
    if end <= start: return None
    try:
        data = json.loads(text[start:end])
        if 'action_type' in data: return PipelineAction(**data)
    except: pass
    return None

def pipeline_reward_fn(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c if isinstance(c, str) else c[0].get('content', '')
        action = parse_action(text)
        if action is None:
            rewards.append(-0.3); continue
        reward = 0.3  # format bonus
        try:
            env = DataPipelineEnv(task_id='hard')
            obs = env.reset()
            result = env.step(action)
            reward += result.reward or 0.0
            if action.action_type == 'compare_schema':
                if result.observation.schema_diff and len(result.observation.schema_diff) > 0:
                    reward += 0.3
        except: reward -= 0.2
        rewards.append(float(reward))
    return rewards

# Sanity check
test_r = pipeline_reward_fn(['{"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 10}}'])
print(f'Reward sanity check: {test_r} (should be > 0)')


Reward sanity check: [0.19999999999999998] (should be > 0)


In [10]:
from src.tasks import TASKS as _available
grpo_task_ids = [t for t in ['hard', 'hard2'] if t in _available]
print(f'GRPO tasks: {grpo_task_ids}')

grpo_prompts = []
for tid in grpo_task_ids:
    for _ in range(25):
        env = DataPipelineEnv(task_id=tid)
        obs = env.reset()
        chat = tokenizer.apply_chat_template(
            [{'role':'system','content':SYSTEM_PROMPT},
             {'role':'user','content':format_obs(obs, 1)}],
            tokenize=False, add_generation_prompt=True)
        grpo_prompts.append({'prompt': chat})

grpo_ds = Dataset.from_list(grpo_prompts)
print(f'GRPO dataset: {len(grpo_ds)} prompts')


GRPO tasks: ['hard']
GRPO dataset: 25 prompts


In [16]:
from trl import GRPOConfig, GRPOTrainer

GRPO_DIR = '/kaggle/working/grpo_qwen'
grpo_config = GRPOConfig(
    report_to="none",
    average_tokens_across_devices=False,
    output_dir=GRPO_DIR,
    num_generations=8,       # G=8 (small model = more samples fit)
    max_completion_length=200,
    temperature=0.8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    beta=0.1,
    loss_type='grpo',
    logging_steps=5, save_steps=50, seed=42,
    max_prompt_length=MAX_SEQ_LENGTH,
    max_grad_norm=1.0, warmup_ratio=0.05,
)

grpo_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    reward_funcs=pipeline_reward_fn,
    args=grpo_config, train_dataset=grpo_ds,
)

print(f'GRPO: KL={grpo_config.beta}, G={grpo_config.num_generations}')
print('Starting GRPO training...')
grpo_stats = grpo_trainer.train()
print(f'GRPO done. Loss: {grpo_stats.training_loss:.4f}')
model.save_pretrained(GRPO_DIR)
tokenizer.save_pretrained(GRPO_DIR)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8
GRPO: KL=0.1, G=8
Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 2 | Total steps = 12
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / pipeline_reward_fn / mean,rewards / pipeline_reward_fn / std
5,0.180296,0.225625,0.048890,30.187500,25.000000,42.000000,0.000000,30.187500,25.000000,42.000000,1.802957,0.225625,0.072591
10,0.131924,0.215313,0.091396,33.562500,27.400000,57.000000,0.000000,33.562500,27.400000,57.000000,1.319236,0.215313,0.117803


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

GRPO done. Loss: 0.1493


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/grpo_qwen/tokenizer_config.json.


('/kaggle/working/grpo_qwen/tokenizer_config.json',
 '/kaggle/working/grpo_qwen/chat_template.jinja',
 '/kaggle/working/grpo_qwen/tokenizer.json')

## 5. Evaluation

In [17]:
from unsloth import FastLanguageModel as FLM
FLM.for_inference(model)

def run_eval(model, tokenizer, task_id, max_steps=20):
    env = DataPipelineEnv(task_id=task_id)
    obs = env.reset(); step = 0
    for step in range(1, max_steps+1):
        if obs.pipeline_passed: break
        msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':format_obs(obs,step)}]
        inp = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(model.device)
        with torch.no_grad():
            out = model.generate(inp, max_new_tokens=200, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        resp = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
        action = parse_action(resp)
        if action is None: action = PipelineAction(action_type='compare_schema', params={'table':'raw_orders'})
        result = env.step(action); obs = result.observation
        if result.done: break
    nt = len(obs.failed_assertions) + len(obs.passed_assertions)
    np_ = len(obs.passed_assertions)
    return {'task_id': task_id, 'score': round(min(max(np_/nt if nt>0 else 0, 0.01), 0.99), 3), 'passed': obs.pipeline_passed, 'steps': step}

eval_tasks = [t for t in ['easy','medium','hard','hard2'] if t in _available]
baseline = {'easy': 0.70, 'medium': 0.55, 'hard': 0.30, 'hard2': 0.30}

print(f"{'Task':<10}{'Base':<10}{'Trained':<10}{'Delta':<10}{'Pass?'}")
print('='*50)
for tid in eval_tasks:
    r = run_eval(model, tokenizer, tid)
    b = baseline.get(tid, 0)
    d = r['score'] - b
    s = '+' if d >= 0 else ''
    print(f"{tid:<10}{b:<10.3f}{r['score']:<10.3f}{s}{d:<9.3f}{'[PASSED]' if r['passed'] else '[FAILED]'}")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Task      Base      Trained   Delta     Pass?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

easy      0.700     0.667     -0.033   [FAILED]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

medium    0.550     0.500     -0.050   [FAILED]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12

hard      0.300     0.250     -0.050   [FAILED]


## 6. Push to Hub (optional)

In [ ]:
# HF_REPO = 'standing-on-giants/data-pipeline-incident-qwen-grpo'
# if HF_TOKEN:
#     print(f'Pushing to {HF_REPO}...')
#     model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
#     print(f'Done: https://huggingface.co/{HF_REPO}')
# else:
#     print('No HF_TOKEN — saved locally at:', GRPO_DIR)
